In [1]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model, clone_model
import numpy as np
from matplotlib import pyplot as plt, patches

from models.ssd_custom import build_model
from models.ssd_transfer_v2 import tl_build_model
from performance_metrics.custom_metric import class_mAP, offset_MAE
from performance_metrics.metrics_classification import precision, recall, f1
from performance_metrics.metrics_regression import mae, mse, root_mse
from loss_function.custom_loss import AOILoss
from custom_layers.GridCenters import GridCenters

from input_encoder_decoder.input_encoder_new import SSDInputEncoder
from input_encoder_decoder.output_decoder import decode_detections
from input_encoder_decoder.data_generator import DataGenerator

%matplotlib inline

2023-10-23 17:18:58.453900: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-23 17:18:58.453942: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-23 17:18:58.453993: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-23 17:18:58.465027: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using TensorFlow backend


2023-10-23 17:19:01.285789: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-23 17:19:01.290228: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2023-10-23 17:19:01.290574: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [3]:
img_height = 256  # Height of the input images
img_width = 256  # Width of the input images
img_channels = 3  # Number of color channels of the input images
intensity_mean = 127.5  # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
intensity_range = 127.5  # Set this to your preference (maybe `None`). The current settings transform the input pixel values to the interval `[-1,1]`.
n_classes = 1  # Number of positive classes
normalize_coords = True  # Whether or not the model is supposed to use coordinates relative to the image size
model_type = 'custom'
build = False
build_variant = 'resnet'

In [4]:
new_metrics = [precision,
               recall,
               f1,
               mae,
               mse,
               root_mse,
               class_mAP,
               offset_MAE]

In [5]:
K.clear_session()

if model_type == 'custom' and build:
    model = build_model(image_size=(img_height, img_width, img_channels),
                        n_classes=n_classes,
                        l2_regularization=0.0005,
                        normalize_coords=normalize_coords,
                        subtract_mean=intensity_mean,
                        divide_by_stddev=intensity_range)

    model.load_weights("saved_trained_models_v2/new_custom_ssd")

elif model_type == 'tl' and build:
    model = tl_build_model(image_size=(img_height, img_width, img_channels),
                           n_classes=n_classes,
                           variant=build_variant,
                           l2_regularization=0.0005,
                           normalize_coords=normalize_coords)
    if build_variant == 'resnet':
        model.load_weights('saved_trained_models_v2/resnetv2_18_backbone_cnn')  # enter saved model path for resnet backbone
    if build_variant == 'mobilenet':
        model.load_weights('saved_trained_models_v2/mobilenetv3_large_backbone_cnn')  # enter saved model path for mobilenet backbone
    if build_variant == 'efficientnet':
        model.load_weights('saved_trained_models_v2/efficientnetv2_b3_backbone_backbone_cnn')  # enter saved model path efficientnet backbone
    if build_variant == 'yolo':
        model.load_weights('saved_trained_models_v2/yolo_v8_m_backbone_backbone_cnn')  # enter saved model path yolo backbone
    

elif not build and model_type == 'custom':
    
    objects = {'GridCenters': GridCenters,
                'compute_loss': AOILoss,
                'f1': f1,
                'mae': mae,
                'mse': mse,
               }
    
    model = load_model('saved_trained_models_v2/new_custom_ssd', custom_objects=objects)  # enter saved model path
    
elif not build and model_type == 'tl':

    objects = {'GridCenters': GridCenters,
                'compute_loss': AOILoss,
                'f1': f1,
                'mae': mae,
                'mse': mse,
               }
    
    if build_variant == 'resnet':
        model = load_model('saved_trained_models_v2/resnetv2_18_backbone_cnn')  # enter saved model path for resnet backbone
    if build_variant == 'mobilenet':
        model = load_model('saved_trained_models_v2/mobilenetv3_large_backbone_cnn')  # enter saved model path for mobilenet backbone
    if build_variant == 'efficientnet':
        model = load_model('saved_trained_models_v2/efficientnetv2_b3_backbone_backbone_cnn')  # enter saved model path efficientnet backbone
    if build_variant == 'yolo':
        model = load_model('saved_trained_models_v2/yolo_v8_m_backbone_backbone_cnn')  # enter saved model path yolo backbone

In [6]:
aoi_loss = AOILoss(neg_pos_ratio=3, alpha=3.0)

model.compile(optimizer='Adam', loss=aoi_loss.compute_loss, metrics=new_metrics)

In [7]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 identity_layer (Lambda)     (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 input_mean_normalization (  (None, 256, 256, 3)          0         ['identity_layer[0][0]']      
 Lambda)                                                                                          
                                                                                                  
 input_stddev_normalization  (None, 256, 256, 3)          0         ['input_mean_normalization

In [8]:
predictor_size = [model.get_layer('conv_loc').output_shape[1:3]]
print('Predictor Layer Dimensions: ', predictor_size)

encoder = SSDInputEncoder(img_height,
                          img_width,
                          n_classes,
                          predictor_sizes=predictor_size,
                          normalize_coords=True,
                          background_id=0)

generator = DataGenerator(parent_dir='/home/kerem/AOI/Datasets/pcb_crops', encoder=encoder, augmentation=True,
                          probability=0.1)

X, y = generator.get_data()

Predictor Layer Dimensions:  [(8, 8)]
Generating image arrays and encoding labels...
Converting images to arrays...


100%|███████████████████████████████████| 15000/15000 [00:12<00:00, 1230.71it/s]


Images as numpy:
(15000, 256, 256, 3)
Parsing ground truth labels from .csv


100%|████████████████████████████████████| 15000/15000 [00:44<00:00, 334.35it/s]


Unencoded labels:
15000
Augmenting images and relabeling...
Applying randomized augmentation...


100%|██████████████████████████████████| 15000/15000 [00:00<00:00, 20900.24it/s]


Encoded labels:
(15000, 64, 12)


In [9]:
model.evaluate(X, y, batch_size=16, verbose=True)

2023-10-23 17:20:04.528055: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2949120000 exceeds 10% of free system memory.
2023-10-23 17:20:06.718977: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2949120000 exceeds 10% of free system memory.


Instructions for updating:
Use fn_output_signature instead


2023-10-23 17:20:10.785616: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700


938/938 [==============================] - 32s 29ms/step - loss: 0.7618 - precision: 0.9521 - recall: 0.9297 - f1: 0.9296 - mae: 0.0082 - mse: 7.1063e-04 - root_mse: 0.0266 - class_mAP: 0.8881 - offset_MAE: 0.0083


[0.7618227601051331,
 0.9520615935325623,
 0.9297264218330383,
 0.9296084642410278,
 0.008237069472670555,
 0.000710627413354814,
 0.026641281321644783,
 0.8880894184112549,
 0.008281526155769825]